# Classification de texte avec des RNN

Dans ces travaux pratiques, nous allons retrouver l'auteur d'une phrase avec des réseaux récurrents. Le corpus est celui des slides : les phrases de 2 œuvres de chacun de 9 auteurs anglophones, de 1821 à 1913 (Brontë, Melville, Wilde…).

1. Du texte aux séquences d'indices, avec `TextVectorization`.
2. Un classifieur récurrent qui ignore le *padding*.
3. L'entraînement d'un GRU bidirectionnel.
4. GRU simple ou bidirectionnel ?
5. Des embeddings GloVe pré-entraînés et figés.
6. Un modèle plus profond.
7. Les auteurs que le modèle confond.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Chaque entraînement y prend de 10 secondes à une minute et quart (le modèle profond de la partie 6) sur une RTX 3090 ; le T4 de Colab est environ deux à trois fois plus lent. Sur CPU, il faudrait d'un quart d'heure à plusieurs heures.

## Téléchargements

Le corpus, le modèle de spaCy pour l'anglais et les vecteurs GloVe de la partie 5 (un fichier de 822 Mo) : comptez moins d'une minute.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Import des bibliothèques

In [ ]:
import pathlib
from collections.abc import Iterable, Iterator

import keras
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import tqdm.auto
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Prétraitements

La cellule suivante prépare les données, comme dans les slides :

- chaque phrase, découpée par spaCy, est un exemple, et son étiquette est son auteur, de 0 à 8 ;
- toute entité nommée (un mot ou un groupe de mots) est remplacée par son type, première lettre triplée pour en faire un mot nouveau (`PPPERSON`, `DDDATE`…) : le modèle doit reconnaître un style, pas les noms des personnages ;
- les dossiers du corpus portent des noms à coquilles (`Melwille`, `bronteCharlote`) : `AUTHORS` donne le nom de chaque auteur ;
- une phrase sans aucun mot (un crochet « [ » isolé par spaCy, de la ponctuation seule) est écartée : vides une fois vectorisées, ces quelque 1 800 phrases seraient entièrement masquées, ce que refuse cuDNN, le noyau des couches récurrentes de Keras sur GPU ;
- 70 % des phrases servent à l'entraînement (`X_train_raw`, `y_train`), 30 % au test (`X_test_raw`, `y_test`).

Elle prend environ une minute.

In [ ]:
# Dossier du corpus → nom de l'auteur
AUTHORS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  # <auteur>/<œuvre>/<chapitre>.txt
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.auto.tqdm(zip(paths, nlp.pipe(contents)), total=len(paths)):
    author = AUTHORS[path.parent.parent.name]
    for sentence in doc.sents:
      text = " ".join(replace_ners(sentence))
      # Une phrase sans lettre ni chiffre serait vide une fois vectorisée
      if any(char.isalnum() for char in text):
        authors.append(author)
        texts.append(text)
  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))
label_encoder = sklearn.preprocessing.LabelEncoder()
y = label_encoder.fit_transform(authors)
print(f"{len(texts)} phrases ; étiquettes 0 à 8 : {', '.join(label_encoder.classes_)}")

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  texts, y, test_size=0.3, random_state=0
)
print(f"Entraînement : {len(X_train_raw)} phrases, test : {len(X_test_raw)}")
for text, label in zip(X_train_raw[:3], y_train[:3]):
  print(f"{label_encoder.classes_[label]} : {text}")

## 1. Du texte aux séquences d'indices

Transformez les phrases en tenseurs d'indices de mots avec une couche `layers.TextVectorization`, `vectorizer` :

- un vocabulaire des 20 000 mots les plus fréquents, appris sur les seules phrases d'entraînement (`adapt`), et rangé dans `vocabulary` ;
- 150 indices par phrase : une phrase plus courte est complétée par des 0 à la fin (le *padding*), une plus longue est tronquée ;
- les phrases d'entraînement transformées dans `X_train`, celles de test dans `X_test`.

Regardez ensuite le résultat : la forme de `X_train`, la première phrase d'entraînement et ses 20 premiers indices, les 10 premiers mots du vocabulaire. Que représentent les indices 0 et 1 ? Quelle proportion des phrases d'entraînement dépasse 150 mots ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.TextVectorization(max_tokens=20_000, output_sequence_length=150)`
- `vectorizer.adapt(X_train_raw)` construit le vocabulaire ; `vectorizer(X_train_raw)` renvoie un tenseur d'entiers de forme `(51246, 150)`
- `vectorizer.get_vocabulary()` : la liste des mots, dans l'ordre de leurs indices
- `len(text.split())` : le nombre de mots d'une phrase

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vectorizer = layers.TextVectorization(max_tokens=..., output_sequence_length=...)
vectorizer.adapt(...)
X_train = vectorizer(...)
X_test = ...
vocabulary = vectorizer.get_vocabulary()

print(X_train.shape)
print(X_train_raw[0])
print(X_train[0, :20].numpy())
print(" | ".join(vocabulary[:10]))
lengths = np.array([len(text.split()) for text in X_train_raw])
print(f"Phrases de plus de 150 mots : {(lengths > ...).mean():.1%}")
```

### Solution

In [ ]:
vectorizer = layers.TextVectorization(
  max_tokens=20_000,  # les 20 000 mots les plus fréquents
  output_sequence_length=150,  # 0 ajoutés à la fin, ou phrase tronquée
)
vectorizer.adapt(X_train_raw)  # vocabulaire de l'entraînement seul
X_train = vectorizer(X_train_raw)
X_test = vectorizer(X_test_raw)
vocabulary = vectorizer.get_vocabulary()

print(X_train.shape)
print(X_train_raw[0])
print(X_train[0, :20].numpy())
print(" | ".join(vocabulary[:10]))
lengths = np.array([len(text.split()) for text in X_train_raw])
print(f"Phrases de plus de 150 mots : {(lengths > 150).mean():.1%}")

# Indice 0 : le padding (la chaîne vide) ; 1 : [UNK], tout mot hors du vocabulaire ;
# ensuite, les mots du plus fréquent au moins fréquent. Moins de 1 % des phrases
# dépassent 150 mots : la troncature ne perd presque rien.

## 2. Un classifieur récurrent qui ignore le *padding*

Écrivez la fonction `build_model(*recurrent_layers, embedding=None)`, qui renvoie un modèle `keras.Sequential` :

- une entrée `keras.Input` : des phrases de toute longueur, faites d'indices entiers ;
- la couche `embedding` si elle est donnée, sinon une nouvelle couche `layers.Embedding` : un vecteur de 64 nombres pour chacun des 20 000 mots du vocabulaire, et le masque du *padding* ;
- les couches récurrentes `recurrent_layers`, dans l'ordre ;
- une couche `layers.Dense` qui donne une probabilité à chacun des 9 auteurs.

Le masque est indispensable ici : une phrase compte 19 mots en médiane, suivis de plus de 130 zéros de *padding*. Sans masque, le RNN les lirait : son état final serait celui du dernier zéro, et la direction arrière d'un RNN bidirectionnel commencerait par eux. Avec un masque, les couches récurrentes sautent ces pas, et leur état final est celui du dernier vrai mot.

Les parties suivantes appellent `build_model` avec la ou les couches récurrentes à comparer, par exemple `build_model(layers.GRU(64))`.

In [ ]:
def build_model(
  *recurrent_layers: layers.Layer, embedding: layers.Embedding | None = None
) -> keras.Sequential:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Input(shape=(None,), dtype="int32")` : `None`, des phrases de toute longueur ; `int32`, des indices
- `layers.Embedding(20_000, 64, mask_zero=True)` : le masque est calculé à partir des indices 0, et transmis aux couches suivantes
- `*recurrent_layers` dans une liste insère les couches une à une
- `layers.Dense(9, activation="softmax")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_model(
  *recurrent_layers: layers.Layer, embedding: layers.Embedding | None = None
) -> keras.Sequential:
  if embedding is None:
    embedding = layers.Embedding(..., ..., mask_zero=...)
  return keras.Sequential(
    [
      keras.Input(shape=..., dtype=...),
      embedding,
      *recurrent_layers,
      layers.Dense(..., activation=...),
    ]
  )
```

### Solution

In [ ]:
def build_model(
  *recurrent_layers: layers.Layer, embedding: layers.Embedding | None = None
) -> keras.Sequential:
  if embedding is None:
    # Un vecteur de 64 nombres par mot, et le masque des indices 0
    embedding = layers.Embedding(20_000, 64, mask_zero=True)
  return keras.Sequential(
    [
      keras.Input(shape=(None,), dtype="int32"),  # phrases de toute longueur
      embedding,
      *recurrent_layers,
      layers.Dense(len(label_encoder.classes_), activation="softmax"),
    ]
  )

## 3. Entraîner un GRU bidirectionnel

La fonction `train` compile un modèle, l'entraîne et l'évalue comme dans les slides :

- perte `sparse_categorical_crossentropy`, puisque les étiquettes sont des entiers ; optimiseur Adam, avec écrêtage de la norme du gradient (`clipnorm`) ;
- des batchs de 256 phrases ; 20 % des phrases d'entraînement en validation ;
- 20 epochs au plus : `EarlyStopping` arrête l'entraînement après 2 epochs sans baisse de la perte de validation, et remet les poids de la meilleure.

Elle affiche les courbes de justesse et renvoie la justesse de test. La cellule suivante l'applique à un GRU bidirectionnel de 64 unités par sens de lecture.

In [ ]:
def train(model: keras.Model, epochs: int = 20) -> float:
  model.compile(
    optimizer=keras.optimizers.Adam(clipnorm=1.0),  # écrêtage du gradient
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  stop = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
  history = model.fit(
    X_train,
    y_train,
    batch_size=256,
    epochs=epochs,
    validation_split=0.2,
    callbacks=[stop],
  )
  _, accuracy = model.evaluate(X_test, y_test, verbose=0)

  # Justesses de la meilleure epoch, celle dont les poids sont gardés
  best = int(np.argmin(history.history["val_loss"]))
  print(
    f"Meilleure epoch : {best + 1} ; justesse d'entraînement "
    f"{history.history['accuracy'][best]:.3f}, "
    f"de validation {history.history['val_accuracy'][best]:.3f}, de test {accuracy:.3f}"
  )
  epochs_run = np.arange(1, len(history.epoch) + 1)
  plt.plot(epochs_run, history.history["accuracy"], label="entraînement")
  plt.plot(epochs_run, history.history["val_accuracy"], label="validation")
  plt.axvline(best + 1, color="gray", linestyle="--", label="meilleure epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Justesse")
  plt.legend()
  plt.show()
  return accuracy

In [ ]:
bigru = build_model(layers.Bidirectional(layers.GRU(64)))
bigru.summary()
bigru_accuracy = train(bigru)

### Premières conclusions

Au vu des courbes et des justesses, que dire de la qualité de cet apprentissage ? Comparez la justesse de test au hasard (une chance sur 9) et à un modèle qui prédirait toujours l'auteur le plus fréquent, Melville (18 % des phrases).

Votre réponse ici

#### Solution

Le modèle apprend : environ 55 % de justesse en test, cinq fois le hasard (11 %) et trois fois la classe majoritaire (18 %).

Mais il surapprend dès la 3e epoch : sa justesse d'entraînement continue de monter (75 % à la 4e epoch), alors que celle de validation plafonne vers 57 % et que la perte de validation remonte. `EarlyStopping` arrête l'entraînement deux epochs plus tard et remet les poids de la meilleure epoch.

Le modèle a 1,3 million de paramètres, dont 1,28 million dans la table d'embeddings, pour 41 000 phrases d'entraînement : il retient les mots des phrases d'entraînement plutôt que le style de chaque auteur.

## 4. GRU simple ou bidirectionnel ?

Entraînez avec `train` un GRU simple de 64 unités, qui ne lit les phrases que dans un sens, et comparez-le au GRU bidirectionnel de la partie 3 :

- la justesse de test ;
- le nombre de paramètres de la couche récurrente, et du modèle entier ;
- la durée d'une epoch, que Keras affiche pendant l'entraînement.

Le second sens de lecture vaut-il son coût ici ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `build_model(layers.GRU(64))` : le même modèle, avec un GRU simple
- `model.summary()` donne le nombre de paramètres de chaque couche ; `model.count_params()` celui du modèle entier
- `bigru_accuracy` contient la justesse de test du GRU bidirectionnel

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
gru = build_model(...)
gru.summary()
gru_accuracy = train(...)
print(f"GRU : {gru_accuracy:.3f}, {gru.count_params()} paramètres")
print(f"GRU bidirectionnel : {...:.3f}, {bigru.count_params()} paramètres")
```

### Solution

In [ ]:
gru = build_model(layers.GRU(64))
gru.summary()
gru_accuracy = train(gru)
print(f"GRU : {gru_accuracy:.3f}, {gru.count_params()} paramètres")
print(f"GRU bidirectionnel : {bigru_accuracy:.3f}, {bigru.count_params()} paramètres")

Le GRU simple fait un peu moins bien : environ 53 % de justesse en test, contre 54 à 56 % pour le GRU bidirectionnel selon les exécutions.

Le second sens de lecture double les paramètres de la couche récurrente (24 960 → 49 920), mais ce n'est que 2 % du modèle, dont l'embedding fait l'essentiel (1,28 million) ; une epoch dure environ 40 % de plus (11 ms par batch contre 8 sur une RTX 3090). Le gain est modeste, mais il vaut ce coût : l'état final d'un GRU simple résume surtout la fin de la phrase, et la direction arrière y ajoute un résumé de son début.

## 5. Des embeddings GloVe pré-entraînés et figés

Les embeddings des modèles précédents partent de vecteurs aléatoires et sont appris sur nos 41 000 phrases d'entraînement. Les vecteurs [GloVe](https://nlp.stanford.edu/projects/glove/), eux, ont été appris sur 6 milliards de mots (Wikipédia et Gigaword) : des mots de sens proche y ont des vecteurs proches.

Construisez une couche `glove_embedding` qui donne à chaque mot de `vocabulary` son vecteur GloVe de 300 dimensions, sans le modifier pendant l'entraînement :

- lisez le fichier `glove.6B.300d.txt` dans un dictionnaire `glove`, mot → vecteur ; chaque ligne est un mot suivi de ses 300 coordonnées, séparés par des espaces, et seuls les mots de `vocabulary` nous intéressent ;
- remplissez `embedding_matrix`, de forme `(len(vocabulary), 300)` : sa ligne `i` est le vecteur du mot `vocabulary[i]`, ou reste nulle si GloVe ne connaît pas ce mot ;
- créez `glove_embedding`, une couche `layers.Embedding` initialisée avec cette matrice, figée, et qui calcule le masque du *padding*.

Combien des 20 000 mots du vocabulaire GloVe connaît-il ? Lesquels manquent ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `word, *coordinates = line.rstrip().split(" ")` sépare le mot de ses coordonnées ; `np.array(coordinates, dtype="float32")` en fait un vecteur
- `set(vocabulary)` : un test `word in ...` rapide sur les 400 000 lignes du fichier
- `layers.Embedding(input_dim=len(vocabulary), output_dim=300, mask_zero=True, embeddings_initializer=keras.initializers.Constant(embedding_matrix), trainable=False)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in ...:
      glove[word] = np.array(coordinates, dtype="float32")

embedding_matrix = np.zeros((len(vocabulary), 300))
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = ...
missing = [word for word in vocabulary if word not in ...]
print(f"{len(glove)} mots sur {len(vocabulary)} dans GloVe")
print("Absents :", ", ".join(missing[:30]))

glove_embedding = layers.Embedding(
  input_dim=...,
  output_dim=...,
  mask_zero=...,
  embeddings_initializer=keras.initializers.Constant(...),
  trainable=...,
)
```

### Solution

In [ ]:
# Seuls les vecteurs des mots du vocabulaire sont gardés
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in words:
      glove[word] = np.array(coordinates, dtype="float32")

embedding_matrix = np.zeros((len(vocabulary), 300))  # 0 si le mot est absent
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = glove[word]
missing = [word for word in vocabulary if word not in glove]
print(f"{len(glove)} mots sur {len(vocabulary)} dans GloVe")
print("Absents :", ", ".join(missing[:30]))

glove_embedding = layers.Embedding(
  input_dim=len(vocabulary),
  output_dim=300,
  mask_zero=True,
  embeddings_initializer=keras.initializers.Constant(embedding_matrix),
  trainable=False,  # table figée
)

# Environ 19 100 des 20 000 mots du vocabulaire sont dans GloVe. Absents : le padding et
# [UNK] (indices 0 et 1), les types d'entités (ppperson, dddate…), des noms que spaCy
# n'a pas reconnus (holgrave, pyncheon), le vocabulaire de la chasse à la baleine
# (harpooneer, whalemen) et des graphies anciennes (oftener, trowsers, noontide).

### Entraîner avec GloVe

Entraînez le même GRU bidirectionnel que dans la partie 3, avec `glove_embedding` pour couche d'embedding, et comparez-le au premier :

- l'écart entre la justesse d'entraînement et celle de validation ;
- la justesse de test ;
- le nombre de paramètres entraînables, que `summary()` donne.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`build_model` prend la couche d'embedding par son paramètre `embedding`, après les couches récurrentes. `bigru_accuracy` contient la justesse de test du premier modèle.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
glove_bigru = build_model(layers.Bidirectional(...), embedding=...)
glove_bigru.summary()
glove_accuracy = train(...)
print(f"Embeddings appris : {bigru_accuracy:.3f}, GloVe figé : {glove_accuracy:.3f}")
```

#### Solution

In [ ]:
glove_bigru = build_model(
  layers.Bidirectional(layers.GRU(64)), embedding=glove_embedding
)
glove_bigru.summary()
glove_accuracy = train(glove_bigru)
print(f"Embeddings appris : {bigru_accuracy:.3f}, GloVe figé : {glove_accuracy:.3f}")

Avec GloVe figé, le modèle n'a plus que 141 705 paramètres entraînables, contre 1,3 million : les 6 millions de nombres de la table GloVe ne bougent pas.

Il surapprend bien plus lentement : sa perte de validation baisse pendant 8 epochs au lieu de 2, jusqu'à un minimum plus bas, et la justesse de test progresse un peu, d'environ 55 % à 57 %. Les vecteurs GloVe apportent ce que nos 41 000 phrases ne peuvent pas apprendre seules : des mots de sens proche y ont des vecteurs proches, même quand ils sont rares dans le corpus.

L'écart entre la justesse d'entraînement et celle de validation finit pourtant par se creuser : la couche récurrente, elle, apprend toujours par cœur. C'est le rôle du *dropout* du modèle suivant.

## 6. Un modèle plus profond

Un dernier modèle, donné : deux GRU bidirectionnels empilés sur les embeddings GloVe.

- `return_sequences=True` : la première couche renvoie un état par mot, la séquence que lit la seconde ;
- `dropout=0.5` : la moitié des valeurs d'entrée de chaque couche récurrente mises à zéro à chaque pas d'entraînement, dans les deux sens de lecture, contre le surapprentissage.

In [ ]:
deep = build_model(
  layers.Bidirectional(layers.GRU(64, dropout=0.5, return_sequences=True)),
  layers.Bidirectional(layers.GRU(64, dropout=0.5)),
  embedding=glove_embedding,
)
deep.summary()
deep_accuracy = train(deep)
print(
  f"GloVe et un GRU bidirectionnel : {glove_accuracy:.3f}, deux : {deep_accuracy:.3f}"
)

Le modèle profond atteint environ 58 % de justesse en test, à peine plus que le GRU bidirectionnel sur GloVe. Le *dropout* a supprimé le surapprentissage : les justesses d'entraînement et de validation restent égales, vers 59 % à la fin. Mais il ralentit l'apprentissage : la perte de validation baisse encore à la 20e epoch, et `EarlyStopping` n'a pas eu à arrêter l'entraînement. Quelques epochs de plus gagneraient sans doute encore un peu.

## 7. Les auteurs que le modèle confond

La cellule suivante affiche la matrice de confusion du modèle profond sur les phrases de test : la ligne d'un auteur donne la répartition de ses phrases entre les auteurs prédits.

In [ ]:
y_pred = deep.predict(X_test, batch_size=1024).argmax(axis=1)
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=label_encoder.classes_,
  normalize="true",  # proportions de chaque ligne
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Auteur prédit")
ax.set_ylabel("Auteur réel")
plt.show()

### Lecture de la matrice

Quels auteurs le modèle reconnaît-il le mieux ? Lesquels confond-il, et qu'ont-ils en commun ? Les œuvres du corpus sont listées dans le [README du jeu de données](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Votre réponse ici

#### Solution

La diagonale donne la part des phrases de chaque auteur bien attribuées ; ces chiffres varient de quelques points d'une exécution à l'autre.

- Wilde (68 %), Melville (65 %) et Thoreau (64 %) sont les mieux reconnus. *Moby-Dick* a un vocabulaire à part (la mer, la chasse à la baleine), les deux livres de Thoreau aussi (la nature, la rivière, les bois), et les pièces de Wilde sont faites de répliques courtes, au style oral, qui ne ressemblent ni aux romans ni aux essais.
- Anne Brontë est la moins bien reconnue (43 %) : 24 % de ses phrases sont attribuées à sa sœur Charlotte. Ce sont deux romancières anglaises de la même époque, aux romans racontés à la première personne, par une gouvernante dans *Agnes Grey* comme dans *Jane Eyre*. La confusion va surtout dans ce sens (7 % des phrases de Charlotte seulement vont à Anne) : Charlotte, presque deux fois plus représentée, attire les phrases ambiguës, et pas seulement celles d'Anne (de 4 à 10 % des phrases de chacun des autres auteurs).
- Shaw et Wilde se confondent : 18 % des phrases de Shaw sont prises pour du Wilde, 13 % de celles de Wilde pour du Shaw. Ce sont les deux dramaturges irlandais du corpus, et leurs répliques se ressemblent.
- Emerson et Thoreau aussi : 22 % des phrases d'Emerson sont attribuées à Thoreau, 12 % dans l'autre sens. Ce sont deux essayistes américains du transcendantalisme, voisins et amis.

Une phrase isolée est un indice faible : beaucoup sont courtes et banales (« Yes . »), et aucun modèle ne peut les attribuer. Pour situer ces résultats, une régression logistique sur une représentation TF-IDF des phrases atteint environ 61 % de justesse sur ce même découpage : sur des phrases courtes, les mots employés comptent plus que leur ordre.